In [1]:
!pip install numba numpy

In [2]:
from numba import cuda
import numpy as np
import math

In [3]:
@cuda.jit
def vector_add(A,B,C):
  idx = cuda.grid(1)
  if(idx < A.size):
    C[idx] = A[idx] + B[idx]

In [4]:
N = 100000
A = np.random.randint(0,100,N).astype(np.float32)
B = np.random.randint(0,100,N).astype(np.float32)
C = np.zeros(N,dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(C)

threads_per_block = 256
blocks_per_grid = math.ceil(N/threads_per_block)


In [5]:
import time
start_cpu = time.perf_counter()

for i in range(N):
  C[i] = A[i] + B[i]

end_cpu = time.perf_counter()

In [6]:
start_gpu = time.perf_counter()
vector_add [blocks_per_grid,threads_per_block] (d_A,d_B,d_C)
cuda.synchronize()
C = d_C.copy_to_host()
end_gpu = time.perf_counter()

In [8]:
print("A[:5]", A[:5])
print("B[:5]", B[:5])
print("C[:5]", C[:5])

print("CPU Execution Time : ", (end_cpu - start_cpu) * 1000, " ms " )
print("GPU Execution Time : ", (end_gpu - start_gpu) * 1000, " ms " )

A[:5] [15. 27. 35. 12. 25.]
B[:5] [20. 53. 35. 97.  8.]
C[:5] [ 35.  80.  70. 109.  33.]
CPU Execution Time :  45.54339999998547  ms 
GPU Execution Time :  3126.5362870000217  ms 
